### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to reperesent the state of conversation when interacting with an LLM. Messages are objects that contains:

- Role -> Identifies the message type (e.g. system, user)
- Content -> Represents the actual content of the message (e.g. text, images, audio, documents)
- Metadata -> Optional fields such as response information, message IDs and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behaviour regardless of the model being called.

In [2]:
## Importing a model and using it to get a response
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [3]:
## Whatever we pass in the invoke method as an argument, it is considered as HumanMessage.
## The response generated will me in the form of AIMessage
response = model.invoke("What are exotic pets")
response

AIMessage(content="<think>\nOkay, so the user is asking about exotic pets. Let me start by recalling what I know about this topic. Exotic pets aren't the usual cats or dogs. They're animals that are not typically kept as pets in a particular area or culture. But I need to be careful here because what's considered exotic can vary by region. For instance, in some countries, a parrot might be common, but in another place, it's exotic.\n\nFirst, I should define exotic pets more precisely. Maybe mention that they can include a wide range of animals like reptiles, birds, small mammals, and even some larger animals. Examples would help here, like snakes, lizards, ferrets, or even more unusual ones like hedgehogs or sugar gliders. But I need to make sure not to list too many examples without context, as the user might be looking for a general definition.\n\nI remember that keeping exotic pets can have legal implications. Different countries have laws about which animals can be kept as pets. Fo

#### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where we don't need to retain the conversation history

We can use Text Prompts when:

- You have single, standalone request
- You don't need conversation history
- You want minimal code complexity

In [ ]:
## Like in this case, the input is treated as HumanMessage, and we can say it is an example of text prompt.
model.invoke("What are exotic pets")

#### Message Prompts

Alternatively, we can pass in a list of messages to the model by providing a list of message objects.

Message types:
- System Message: Tells the model how to behave and provide context for interactions
- Human Message: Represent user input and interactions with models
- AI Message: Responses generated by the model, including text content, tool calls and metadata
- Tool Message: Represents the output of tool calls

In [4]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on Artificial Intelligence")
]

response = model.invoke(messages)
response.content

'<think>\nOkay, the user wants a poem about Artificial Intelligence. Let me start by thinking about the key aspects of AI. There\'s the creation angle, how humans built it. Maybe mention different applications like healthcare, tech, etc. Also, the duality of AI—its potential for good and the risks. I need to balance the positive and the cautionary elements.\n\nFirst, establish the birth of AI, maybe use metaphors like circuits or code. Then talk about its growth and learning. Use imagery related to light or fire for progress. Mention specific fields it impacts. Then shift to the potential downsides, the ethical concerns. Use contrasting words like "shadow" or "whispers of doubt." Conclude with a hopeful message about human responsibility. Make sure the poem flows smoothly with a consistent rhyme scheme. Maybe use quatrains with alternating rhymes. Check for rhythm and readability. Avoid being too technical so it\'s accessible. Use vivid language to evoke emotions. Let me try drafting s

In [5]:
## Detailed info to the LLM through SystemMessage
system_message = SystemMessage("""
    You are a senior Python developer with an expertise in web frameworks. 
    Always provide code example and explain your reasoning.
    Be concise but thorough in your explanations
""")

messages = [
    system_message,
    HumanMessage("How do I create a REST API?")
]

response = model.invoke(messages)
response.content

'<think>\nOkay, the user is asking how to create a REST API. Let me start by recalling what a REST API is. REST stands for Representational State Transfer, and it\'s an architectural style for designing networked applications. It uses HTTP methods like GET, POST, PUT, DELETE, etc., to perform operations on resources.\n\nThe user is probably looking for a step-by-step guide. Since they mentioned they want code examples and explanations, I should use a Python web framework. The most common ones are Flask and Django. Flask is lightweight and good for small APIs, while Django has more built-in features but might be overkill for simple APIs. Let\'s go with Flask because it\'s simpler for an example.\n\nFirst, they need to install Flask. So the first step is to install it using pip. Then, set up a basic Flask application. The example should include defining endpoints and HTTP methods.\n\nLet me outline the steps. Create a new Python file, import Flask, create an instance. Then define a route

In [7]:
## Message Metadata
human_msg = HumanMessage(
    content = "Hi, how are you?",
    name = "Anushk", # Optional: to identify the users
    id = "msg_124" # Optional: unique identifier for tracing
)

response = model.invoke([human_msg])
response

AIMessage(content='<think>\nOkay, the user said "Hi, how are you?" I need to respond warmly. First, I should greet them back and ask about their well-being to establish a friendly conversation. I\'ll keep it simple and natural. Since they might want to share their feelings, I\'ll leave room for them to talk. I should use casual language, maybe add an emoji to make it more approachable. Let me check if there\'s anything else I need to consider. No, that\'s good. I\'ll make sure the tone is positive and supportive.\n</think>\n\nHi there! ٩(◕‿◕｡)۶ I\'m doing well, thanks for asking! How about you? I\'d love to hear how you\'re feeling today. Let\'s chat!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 14, 'total_tokens': 171, 'completion_time': 0.362632185, 'completion_tokens_details': None, 'prompt_time': 0.000559492, 'prompt_tokens_details': None, 'queue_time': 0.185041377, 'total_time': 0.363191677}, 'model_name': 'qwen/qwen3-32b', 

In [8]:
## Another example where we create the AIMessage on our own
ai_msg = AIMessage("I'd be happy to assist you with that question")

messages = [
    SystemMessage("You are an helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("Great! What's 300*20?")
]

response = model.invoke(messages)
response

AIMessage(content="<think>\nOkay, let's see. The user asked for 300 multiplied by 20. Hmm, I need to calculate that. Let me break it down. Multiplying by 20 is the same as multiplying by 10 and then doubling it, right? So first, 300 times 10 is 3,000. Then, doubling that gives 3,000 plus 3,000, which is 6,000. Wait, is there another way to check that? Maybe by breaking down 300 into 3*100. So 3*100*20. That's 3*(100*20) = 3*2,000 = 6,000. Yeah, that seems right. Let me make sure there's no mistake. 300*20. If I add 300 twenty times, but that's tedious. Alternatively, using place value: 300 is 3 hundreds, times 20 (which is 2 tens). So multiplying 3 hundreds by 2 tens gives 6 thousand. So 6,000. Yep, all methods point to the same answer. I think that's correct.\n</think>\n\nSure! \n\n**300 multiplied by 20 is 6,000.**  \n\nYou can break it down like this:  \n$ 300 \\times 20 = 300 \\times (2 \\times 10) = (300 \\times 2) \\times 10 = 600 \\times 10 = 6,000 $.  \n\nLet me know if you nee

In [9]:
## An example using the ToolMessage
from langchain.messages import ToolMessage

# After a model makes a tool call, here we are doing all that manually
ai_msg = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "New Delhi"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "ThunderStorm & Rain"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id = "call_123", # Must match the callID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in New Delhi"),
    ai_msg, # Model's tool call
    tool_message # Tool execution result
]

model.invoke(messages)

AIMessage(content='<think>\nOkay, the user asked for the weather in New Delhi. I called the get_weather function with the location set to New Delhi. The response came back as "ThunderStorm & Rain". Now I need to present this information clearly.\n\nFirst, I should mention the current weather condition. Since it\'s a thunderstorm with rain, that\'s important to highlight. People might want to know if it\'s safe to go out. I should also add a note about staying safe, like avoiding flooded areas and staying indoors if possible. Maybe include some practical advice, such as carrying an umbrella or wearing a raincoat. Keep the response friendly and helpful, making sure the user gets the necessary information to plan accordingly.\n</think>\n\nThe current weather in New Delhi is a **thunderstorm with rain**. It\'s advisable to stay indoors, avoid flooded areas, and carry an umbrella or raincoat if you must step out. Stay safe! 🌧️⛈️', additional_kwargs={}, response_metadata={'token_usage': {'co